In [ ]:
!pip install flask-cors pyngrok accelerate bitsandbytes peft transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


In [ ]:
# Force upgrade essential libraries
!pip install -U bitsandbytes transformers accelerate peft flask-cors pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 40.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1


In [ ]:
!ngrok config add-authtoken 389fB4BAG9t6Ys5EWEZh0M5cxMV_4xyXZfmQwwnhH4a1Mf8CK

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


Load Only Medical Model

In [ ]:
!ls /content/models

llama-med-adapter


In [ ]:
import os
print(os.listdir("/content/models/llama-med-adapter"))

['special_tokens_map.json', 'README.md', 'tokenizer.model', 'tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json']


In [ ]:
from pyngrok import ngrok
ngrok.kill()

In [ ]:
import torch, os, gc, threading
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import AutoModelForCausalLM, LlamaTokenizer, BitsAndBytesConfig
from peft import PeftModel
from pyngrok import ngrok

# --- CLEANUP ---
def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()

clear_vram()

# --- CONFIG ---
model_id = "openlm-research/open_llama_7b_v2"
med_path = "/content/models/llama-med-adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading Model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = LlamaTokenizer.from_pretrained(model_id)

# ✅ Load ONLY medical adapter
model = PeftModel.from_pretrained(base_model, med_path)

# --- FLASK APP ---
app = Flask(__name__)
CORS(app)

@app.route("/chat", methods=["POST"])
def chat():
    try:
        data = request.json
        query = data.get("prompt", "").strip()

        # Fixed medical instruction
        instruction = "Answer the medical question accurately and concisely."

        prompt = f"### Instruction: {instruction}\n### Question: {query}\n### Answer:"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        input_len = inputs.input_ids.shape[1]

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=True,
                temperature=0.3,
                top_p=0.9,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id
            )

        raw_answer = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()
        clean_answer = raw_answer.split("###")[0].strip()

        return jsonify({"response": clean_answer})

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# --- NGROK ---
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)

    public_url = ngrok.connect(5000).public_url
    print(f"\n✅ SERVER READY")
    print(f"🔗 API URL: {public_url}/chat")

except Exception as e:
    print(f"Ngrok Error: {e}")

threading.Thread(
    target=lambda: app.run(port=5000, debug=False, use_reloader=False)
).start()

Loading Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/512k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/330 [00:00<?, ?B/s]


✅ SERVER READY
🔗 API URL: https://striped-carmelo-unadoringly.ngrok-free.dev/chat


In [ ]:
import requests

# UPDATE THIS URL with the one printed by the server code above
url = "https://striped-carmelo-unadoringly.ngrok-free.dev/chat"

data = {
    "prompt": "what is fever ?",
    "model_type": "medical"
}

print("Sending request to Llama... (please wait)")
try:
    response = requests.post(url, json=data)

    if response.status_code == 200:
        print("\n--- AI RESPONSE ---")
        print(response.json().get("response"))
    else:
        print(f"Error: Server returned status {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"Connection Error: {e}")

Sending request to Llama... (please wait)


INFO:werkzeug:127.0.0.1 - - [22/Apr/2026 13:05:08] "POST /chat HTTP/1.1" 200 -



--- AI RESPONSE ---
Leukaemias are cancers of white blood cells, which help fight infection by producing antibodies to destroy foreign invaders such as bacteria or viruses in your body's tissues (such infections cause diseases called bacterial pneumoniaor viral hepatitis).  Leukeamia can be divided into two main types based on how quickly it develops - acute lymphocyticleuekemicaparison with other formsof canceris that they tend not onlyto grow very fast but also spread throughoutthe wholebody rapidly; this means treatment must begin early before symptoms appearand may include chemotherapy(treatment using drugs)radiation therapya type of surgerycalled bone marrow transplantationinvolving a donor stem cellsthat will replace
